In [174]:
import requests
import pandas as pd
import numpy as np

In [175]:
movie_master = pd.read_excel("../Dataset/Movie_Master.xlsx")
movie_master

,MOVIE_ID,TITLE,RELEASE_MONTH
0,0,걸어서하늘까지(1992),202103
1,1,너와극장에서,201807
2,2,가려진시간[가치봄],202010
3,3,그링고,202003
4,4,스위치(2010),202010
...,...,...,...
14013,14593,영웅:샐러멘더의비밀[감독판],201309
14014,14594,더로버,201505
14015,14595,메디엄,201410
14016,14596,그린북,201903


In [176]:
api_key = "be60b045f84d63636614fa35a74fc312"


In [177]:
temp = []

In [ ]:
import os
import time

def safe_get(url, retries=3, delay=2):
	for i in range(retries):
		try:
			return requests.get(url, timeout=10).json()
		except Exception as e:
			print(f"  재시도 {i+1}/{retries}: {e}")
			time.sleep(delay)
	return {}

if os.path.exists("movie_crawling_temp.csv"):
	existing = pd.read_csv("movie_crawling_temp.csv", encoding="utf-8-sig")
	temp = existing.to_dict("records")
	done_ids = set(existing["MOVIE_ID"])
	print(f"기존 데이터 {len(temp)}개 로드 완료")
else:
	temp = []
	done_ids = set()
	print("새로 시작")

try:
	for _, row in movie_master.iterrows():
		movie_name = row["TITLE"]
		movie_id = row["MOVIE_ID"]

		if movie_id in done_ids:
			continue

		time.sleep(0.3)

		movieListURL = f"http://kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieList.json?key={api_key}&movieNm={movie_name}"
		movieList = safe_get(movieListURL)

		if "movieListResult" not in movieList:
			temp.append({"MOVIE_ID": movie_id, "totCnt": None, "MovieCd": None, "MovieNm(csv)": movie_name})
			done_ids.add(movie_id)
			print(f"[{movie_id}] {movie_name} → API 응답 오류: {movieList}")
			continue

		totCnt = movieList["movieListResult"]["totCnt"]

		if totCnt == 1:
			movieCd = movieList["movieListResult"]["movieList"][0]["movieCd"]
			movieInfoURL = f"http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieInfo.json?key={api_key}&movieCd={movieCd}"
			movieInfo = safe_get(movieInfoURL)

			if "movieInfoResult" not in movieInfo:
				temp.append({"MOVIE_ID": movie_id, "totCnt": totCnt, "MovieCd": movieCd, "MovieNm(csv)": movie_name})
				done_ids.add(movie_id)
				print(f"[{movie_id}] {movie_name} → movieInfo 응답 오류: {movieInfo}")
				continue

			temp.append(
				{
					"MOVIE_ID"		: movie_id,
					"totCnt"		: totCnt,
					"MovieCd"		: movieCd,
					"MovieNm(csv)"	: movie_name,
					"movieNm(api)"	: movieInfo["movieInfoResult"]["movieInfo"]["movieNm"],
					"showTm"		: movieInfo["movieInfoResult"]["movieInfo"]["showTm"],
					"prdtYear"		: movieInfo["movieInfoResult"]["movieInfo"]["prdtYear"],
					"openDt"		: movieInfo["movieInfoResult"]["movieInfo"]["openDt"],
					"typeNm"		: movieInfo["movieInfoResult"]["movieInfo"]["typeNm"],
					"nationNm"		: movieInfo["movieInfoResult"]["movieInfo"]["nations"][0]["nationNm"] if movieInfo["movieInfoResult"]["movieInfo"]["nations"] else None,
					"genreNm"		: movieInfo["movieInfoResult"]["movieInfo"]["genres"][0]["genreNm"] if movieInfo["movieInfoResult"]["movieInfo"]["genres"] else None,
					"watchGrade"	: movieInfo["movieInfoResult"]["movieInfo"]["audits"][0]["watchGradeNm"] if movieInfo["movieInfoResult"]["movieInfo"]["audits"] else None
				}
			)
			print(f"[{movie_id}] {movie_name} → {movieInfo['movieInfoResult']['movieInfo']['movieNm']} 저장")

		else:
			temp.append(
				{
					"MOVIE_ID"		: movie_id,
					"totCnt"		: totCnt,
					"MovieCd"		: None,
					"MovieNm(csv)"	: movie_name,
				}
			)
			print(f"[{movie_id}] {movie_name} → 검색결과 {totCnt}건 (예외처리)")

		done_ids.add(movie_id)

		if len(temp) % 50 == 0:
			pd.DataFrame(temp).to_excel("movie_crawling_temp.csv", index=False, encoding="utf-8-sig")

finally:
	pd.DataFrame(temp).to_excel("movie_crawling_temp.csv", index=False, encoding="utf-8-sig")
	print(f"저장 완료. 총 {len(temp)}개")


기존 데이터 6153개 로드 완료
[6389] 성범죄수사대 → 검색결과 2건 (예외처리)
[6390] 리플레이 → 검색결과 16건 (예외처리)
[6391] 어바웃웨딩 → 검색결과 7건 (예외처리)
[6392] 귀수동화 → 귀수동화 저장
[6393] 더시그널(2007) → 검색결과 0건 (예외처리)
[6394] 미션:톱스타를훔쳐라 → 미션: 톱스타를 훔쳐라 저장
[6395] 싸이코(1998) → 검색결과 0건 (예외처리)
[6396] 어스인베이젼 → 어스 인베이젼 저장
[6397] 착신아리1 → 검색결과 0건 (예외처리)
[6398] 안시성[가치봄] → 검색결과 0건 (예외처리)
[6399] 오퍼레이션레드씨 → 오퍼레이션 레드 씨 저장
[6400] 열정의제국[리마스터링] → 검색결과 0건 (예외처리)
[6402] 절대연필 → 검색결과 3건 (예외처리)
[6403] 더블데이트대소동 → 검색결과 3건 (예외처리)
[6404] 황비홍-전설의묵룡도를찾아서 → 검색결과 0건 (예외처리)
[6405] 가나의혼인잔치:언약 → 갈릴리 예수 저장
[6406] 장마(2019) → 검색결과 0건 (예외처리)
[6407] 한강블루스 → 한강블루스 저장
[6409] 익스트림게임:서울어택 → 익스트림 게임 : 서울어택 저장
[6410] 성난얼굴로돌아보라 → 검색결과 2건 (예외처리)
[6411] 알파테스트 → 알파 테스트 저장
[6412] 트랜스포머:사라진시대 → 트랜스포머: 사라진 시대 저장
[6413] 라푼젤 → 검색결과 2건 (예외처리)
[6414] 페르소나(1966) → 검색결과 0건 (예외처리)
[6415] 스타트렉10-네미시스 → 검색결과 0건 (예외처리)
[6416] 데드맨1부:복수의서막 → 데드맨 1부: 복수의 서막 저장
[6417] 어롱웨이다운 → 어 롱 웨이 다운 저장
[6418] 백만거악 → 백만거악 저장
[6419] 생존게임화씨247 → 검색결과 0건 (예외처리)
[6420] 몬스터캅 → 몬스터 캅 저장
[6421] 어폴로지 → 어폴로지 저장
[6422] 괜찮아! 

KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame(temp)
df

,MOVIE_ID,totCnt,MovieCd,MovieNm(csv),movieNm(api),showTm,prdtYear,openDt,typeNm,nationNm,genreNm,watchGrade
0,0,0.0,20156921,걸어서하늘까지(1992),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1.0,20175865,너와극장에서,너와 극장에서,79.0,2017.0,20180628.0,옴니버스,한국,드라마,12세이상관람가
2,2,0.0,20175865,가려진시간[가치봄],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,6.0,20175865,그링고,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,0.0,20175865,스위치(2010),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
14013,14593,NaN,None,영웅:샐러멘더의비밀[감독판],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14014,14594,NaN,None,더로버,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14015,14595,NaN,None,메디엄,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14016,14596,NaN,None,그린북,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
